## 3. Подготовка данных

In [965]:
import pandas as pd
import ast
import re
import numpy as np

In [966]:
clean_data = pd.read_csv('housing_data.csv')

#### 3.2 Создадим новую колонку `privatePool` на основе колонок `private pool` и `PrivatePool`

In [967]:
clean_data['PrivatePool'] = clean_data['PrivatePool'].replace({"yes": "Yes"})
clean_data['PrivatePool'].value_counts()

PrivatePool
Yes    40311
Name: count, dtype: int64

In [968]:
clean_data['privatePool'] = clean_data[['private pool', 'PrivatePool']].apply(
    lambda x: 'Yes' if 'Yes' in x.values else (np.nan if x.isna().all() else 'No'), axis=1
)
clean_data['privatePool'].value_counts()

privatePool
Yes    44492
Name: count, dtype: int64

In [969]:
clean_data = clean_data.drop(columns=['private pool', 'PrivatePool'])
clean_data.shape

(377185, 17)

#### 3.3 Очистим значения для колонки `status`

In [970]:
clean_data['status'] = clean_data['status'].astype(str).str.lower()
clean_data['status'].value_counts()

status
for sale                199571
active                  105207
nan                      39918
foreclosure               6769
new construction          5475
                         ...  
coming soon: dec 25.         1
coming soon: oct 24.         1
pending take backups         1
contract                     1
coming soon: dec 23.         1
Name: count, Length: 153, dtype: int64

In [971]:
for_sales_statuses = [
    'for sale', 'active', 'a active', 'active option', 'temporary active',
    'price change', 'back on market', 'listing extended', 're activated', 'reactivated'
]
still_show = [s for s in clean_data['status'].unique() if 'show' in s and 'no' not in s]
coming_soon = [s for s in clean_data['status'].unique() if 'coming' in s]
for_sales_statuses = for_sales_statuses + coming_soon + still_show
clean_data['status'] = clean_data['status'].replace(for_sales_statuses, 'for sale')

In [972]:
foreclosure = [s for s in clean_data['status'].unique() if 'forecl' in s or 'aucti' in s]
clean_data['status'] = clean_data['status'].replace(foreclosure, 'foreclosure')

In [973]:
new_status = [s for s in clean_data['status'].unique() if 'new' in s]
clean_data['status'] = clean_data['status'].replace(new_status, 'new')

In [974]:
valid_statuses = ['for sale', 'foreclosure', 'new', 'nan']
clean_data = clean_data[clean_data['status'].isin(valid_statuses)]
clean_data['status'] = clean_data['status'].replace('nan', np.nan)

In [975]:
print(clean_data['status'].value_counts(), clean_data.shape)

status
for sale       308369
foreclosure     12419
new              6165
Name: count, dtype: int64 (366871, 17)


#### 3.4 Очистим значения для колонки `propertyType`

In [976]:
clean_property: pd.DataFrame = clean_data.copy()
clean_property['propertyCategory'] = clean_property['propertyType']

In [977]:
clean_property['propertyCategory'] = clean_property['propertyCategory'].astype(str).str.lower()
print('None values: ', clean_property['propertyCategory'].isna().sum())
print(clean_property['propertyCategory'].value_counts())

None values:  0
propertyCategory
single-family home                                             91737
single family                                                  62833
condo                                                          42450
nan                                                            34645
single family home                                             25288
                                                               ...  
contemporary/modern, french, mediterranean, traditional            1
contemporary, farmhouse                                            1
custom, elevated, other                                            1
1 story, contemporary, traditional, mediterranean                  1
bilevel, converted dwelling, loft with bedrooms, condo/unit        1
Name: count, Length: 1269, dtype: int64


In [978]:
single_family = [s for s in clean_property['propertyCategory'].unique() if
                 'single' in s and 'family' in s]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(single_family, 'single_family')

In [979]:
single_detached = [s for s in clean_property['propertyCategory'].unique() if
                   'detached' in s and 'single' in s]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(single_detached, 'single_family')

In [980]:
detached = [s for s in clean_property['propertyCategory'].unique() if
            'detached' in s and ('manufactured' not in s and 'duplex' not in s)]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(detached, 'single_family')

In [981]:
condo_apt = [s for s in clean_property['propertyCategory'].unique() if
             "condo" in s or "apartment" in s or "high rise" in s or "loft" in s or 'high-rise' in s
             or 'mid-rise' in s or 'low-rise' in s]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(condo_apt, 'condo_apartment')

In [982]:
multi_family = [s for s in clean_property['propertyCategory'].unique() if
                "plex" in s or ('multi' in s and 'family' in s) or "multiple occupancy" in s]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(multi_family, 'multi_family')

In [983]:
townhouse = [s for s in clean_property['propertyCategory'].unique() if "townhouse" in s]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(townhouse, 'townhouse')

In [984]:
land = [s for s in clean_property['propertyCategory'].unique() if
        ("land" in s or 'lot' in s or "vacant" in s) and 'new englander' not in s]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(land, 'land_lot')

In [985]:
coop = [s for s in clean_property['propertyCategory'].unique() if
        "coop" in s or 'co-op' in s or "co op" in s]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(coop, 'co_op')

In [986]:
mobile = [s for s in clean_property['propertyCategory'].unique() if
          "mobile" in s or 'manufactured' in s or "modular" in s]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(mobile, 'mobile_home')

In [987]:
extra_single_family = [
    s for s in clean_property['propertyCategory'].unique() if
    "traditional" in s or 'ranch' in s or "contemporary" in s or "colonial" in s
    or 'transitional' in s or 'bungalow' in s or 'garden home' in s or "story" in s or 'stories' in s or "mediterranean" in s or 'craftsman' in s or "victorian" in s or "split-level" in s or 'cluster home' in s
]
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace(extra_single_family, 'single_family')

In [988]:
valid = [
    "single_family",
    "condo_apartment",
    "nan",
    "land_lot",
    "townhouse",
    "multi_family",
    "co_op",
    "mobile_home"
]

clean_property['propertyCategory'] = clean_property['propertyCategory'].apply(
    lambda x: x if str(x) in valid else "other"
)
clean_property['propertyCategory'] = clean_property['propertyCategory'].replace('nan', np.nan)

In [989]:
clean_property['propertyCategory'].value_counts()

propertyCategory
single_family      206902
condo_apartment     52868
land_lot            30897
townhouse           18489
multi_family        12109
co_op                3836
other                3707
mobile_home          3418
Name: count, dtype: int64

In [990]:
clean_data['propertyCategory'] = clean_property['propertyCategory']

#### 3.5 Очистим значения для колонки `baths`

In [991]:
clean_bath: pd.DataFrame = clean_data.copy()

In [992]:
def parse_rooms(val):
    if pd.isna(val):
        return np.nan
    # Извлекаем первое число (целое или дробное)
    match = re.search(r"\d+(\.\d+)?", str(val).replace(',', ''))
    if match:
        return float(match.group())
    return np.nan


clean_bath['baths_num'] = clean_bath['baths'].apply(parse_rooms)

In [993]:
clean_bath.loc[
    ((clean_bath['baths_num'] % 0.5 != 0) | (clean_bath['baths_num'] == 0)),
    'baths_num'
] = np.nan

In [994]:
clean_bath['baths_num'].value_counts()

baths_num
2.0      97749
3.0      63991
4.0      25842
1.0      15400
2.5      11250
         ...  
14.5         1
116.0        1
29.0         1
241.0        1
68.0         1
Name: count, Length: 74, dtype: int64

In [995]:
clean_data['baths_num'] = clean_bath['baths_num']

#### 3.6 Очистим значения для колонки `beds`

In [996]:
clean_beds: pd.DataFrame = clean_data.copy()

In [997]:
clean_beds['beds_num'] = clean_beds['beds'].apply(parse_rooms)

In [998]:
clean_beds.loc[((clean_beds['beds_num'] % 1 != 0) | (clean_beds['beds_num'] == 0)), 'beds_num'] = np.nan

In [999]:
clean_beds['beds_num'].value_counts()

beds_num
3.0         101710
4.0          66700
2.0          48355
5.0          21165
6.0           6461
             ...  
360731.0         1
5488.0           1
9365.0           1
5654.0           1
8479.0           1
Name: count, Length: 652, dtype: int64

In [1000]:
clean_data['beds_num'] = clean_beds['beds_num']

#### 3.7 Очистим значения для колонки `sqft`

In [1001]:
clean_sqft: pd.DataFrame = clean_data.copy()

In [1002]:
def parse_sqft(val):
    if pd.isna(val):
        return np.nan
    # убираем текст, пробелы, запятые
    val = str(val).lower().replace('sqft', '').replace(',', '').strip()
    # если пусто или '--' -> NaN
    if val in ['', '--', 'nan']:
        return np.nan
    # извлекаем число
    match = re.search(r"\d+", val)
    if match:
        return int(match.group())
    return np.nan


# Применяем преобразование
clean_sqft['sqft_num'] = clean_sqft['sqft'].apply(parse_sqft)

# Заменяем нули на NaN
clean_sqft.loc[clean_sqft['sqft_num'] == 0, 'sqft_num'] = np.nan

In [1003]:
clean_sqft['sqft_num'].value_counts()

sqft_num
1200.0     1392
1000.0     1006
1500.0      985
1800.0      951
1100.0      923
           ... 
20816.0       1
7126.0        1
20518.0       1
8155.0        1
13870.0       1
Name: count, Length: 9859, dtype: int64

In [1004]:
clean_data['sqft_num'] = clean_sqft['sqft_num']

#### 3.8 Очистим значения для колонки `fireplace`

In [1005]:
clean_fireplace = clean_data.copy()

In [1006]:
clean_fireplace['fireplace'] = clean_fireplace['fireplace'].astype(str).str.lower()
clean_fireplace['fireplace'] = clean_fireplace['fireplace'].replace('nan', np.nan)
clean_fireplace['fireplace'].value_counts()

fireplace
yes                                                                                               70941
1                                                                                                 13141
2                                                                                                  2124
not applicable                                                                                     1992
fireplace                                                                                           592
                                                                                                  ...  
2 fireplace, fireplace family rm, fireplace living rm, fireplace master bdr, two way fireplace        1
gas, wood burning, two, propane logs convey                                                           1
one, living room                                                                                      1
familyrm, great room, living room                     

In [1007]:
no_fireplace = [s for s in clean_fireplace['fireplace'].unique() if pd.notna(s) and (
        s == "no" or s == "0" or "no fireplace" in s or s == "not applicable")]
clean_fireplace['fireplace_present'] = clean_fireplace['fireplace'].apply(
    lambda x: 'No' if x in no_fireplace else ('Yes' if pd.notna(x) else np.nan)
)
clean_fireplace['fireplace_present'].value_counts()

fireplace_present
Yes    96980
No      2555
Name: count, dtype: int64

In [1008]:
TYPE_PATTERNS = {
    "gas":      re.compile(r"\b(gas|gas\s*log(s)?|gas[-\s]*fp|propane|lp\b|natural\s+gas)\b"),
    "wood":     re.compile(r"\b(wood(\s*burn(ing)?)?|wood\s*stove)\b"),
    "electric": re.compile(r"\b(electric)\b"),
    "pellet":   re.compile(r"\b(pellet(\s*stove)?)\b"),
    "coal":     re.compile(r"\b(coal)\b"),
}

def get_fireplace_type_row(row) -> str:
    """Определяет тип камина, используя колонку fireplace_present (yes/no)."""
    pres = row.get("fireplace_present")
    if pd.isna(pres) or pd.isna(row.get("fireplace")):
        return np.nan

    pres_norm = str(pres).strip().lower()
    if pres_norm == "no":
        return "none"

    # pres == "yes": пробуем распознать тип по тексту fireplace
    txt = str(row["fireplace"]).lower()
    found = [name for name, pat in TYPE_PATTERNS.items() if pat.search(txt)]

    if len(found) == 1:
        return found[0]
    if len(found) > 1:
        return "multiple"
    return "unknown"

In [1009]:
clean_fireplace["fireplace_type"] = clean_fireplace.apply(get_fireplace_type_row, axis=1)
clean_fireplace["fireplace_type"].value_counts()

fireplace_type
unknown     92483
gas          2579
none         2555
wood         1503
multiple      254
electric      151
pellet          9
coal            1
Name: count, dtype: int64

In [1010]:
LOC_PATTERNS = {
    "family room":   re.compile(r"\bfamily\s*(room|rm)\b"),
    "great room":    re.compile(r"\bgreat\s*(room|rm)\b"),
    "living room":   re.compile(r"\bliving\s*(room|rm)\b"),
    "primary bedroom": re.compile(r"\b(primary|owner'?s?|master)\s*(bed(room|rm)?|suite)\b"),
    "bedroom":       re.compile(r"\bbed(room|rm)s?\b"),
    "dining room":   re.compile(r"\bdining\s*(room|rm)\b"),
    "kitchen":       re.compile(r"\bkitchen\b"),
    "den/study/office": re.compile(r"\b(den|study|office|library)\b"),
    "basement":      re.compile(r"\b(basement|lower\s+level)\b"),
    "loft":          re.compile(r"\bloft\b"),
    "bonus room":    re.compile(r"\bbonus\s*(room|rm)?\b"),
    "rec room":      re.compile(r"\b(recreation|rec|game)\s*(room|rm)\b"),
    "sunroom":       re.compile(r"\bsun\s*room\b|\bsunroom\b"),
    "porch":         re.compile(r"\bporch\b"),
    "deck":          re.compile(r"\bdeck\b"),
    "patio/outdoor": re.compile(r"\b(outdoor|outside|patio|lanai|courtyard|terrace|veranda?h?|balcony)\b"),
    "media room":    re.compile(r"\bmedia\s*(room|rm)\b|home\s*theat(re|er)"),
    "garage":        re.compile(r"\bgarage\b"),
    "keeping room":  re.compile(r"\bkeeping\s*room\b"),
    "sitting room":  re.compile(r"\bsitting\s*(room|rm)\b"),
}

def get_fireplace_location_row(row) -> str:
    """
    Определяет локацию камина, используя колонку fireplace_present (yes/no).
    - no  -> 'none'
    - yes -> ищем ключевые слова в fireplace
    - NaN -> NaN
    """
    pres = row.get("fireplace_present")
    if pd.isna(pres) or pd.isna(row.get("fireplace")):
        return np.nan

    pres_norm = str(pres).strip().lower()
    if pres_norm == "no":
        return "none"

    txt = str(row["fireplace"]).lower()

    found = []
    for name, pat in LOC_PATTERNS.items():
        if pat.search(txt):
            found.append(name)

    if len(found) == 0:
        return "unknown"
    if len(found) == 1:
        return found[0]
    return "multiple"

In [1011]:
clean_fireplace["fireplace_location"] = clean_fireplace.apply(get_fireplace_location_row, axis=1)
clean_fireplace["fireplace_location"].value_counts()

fireplace_location
unknown             93078
none                 2555
living room          1220
family room          1114
great room            728
multiple              676
den/study/office      106
basement               17
bedroom                13
rec room                7
patio/outdoor           7
kitchen                 4
dining room             4
bonus room              4
keeping room            1
porch                   1
Name: count, dtype: int64

In [1012]:
WORDS2NUM = {
    "zero":"0","one":"1","two":"2","three":"3","four":"4",
    "five":"5","six":"6","seven":"7","eight":"8","nine":"9","ten":"10"
}

ONE_FP_HINT = re.compile(r"\b(double[-\s]?sided|see[-\s]?through|two[-\s]?way|3[-\s]?sided)\b")

def get_fireplace_count(row):
    """
    Итоговое количество каминов.
    Логика:
      - fireplace_present == 'no'  -> 0
      - fireplace_present == 'yes' -> сначала пробуем достать число из fireplace;
        если нет числа: считаем по локациям; иначе применяем эвристики.
      - fireplace_present is NaN   -> NaN
    Параметры:
      cap: верхняя отсечка (None, чтобы не ограничивать)
      return_int: вернуть целое (Int64), если True
    """
    pres = row.get("fireplace_present")
    if pd.isna(pres):
        return np.nan

    pres_norm = str(pres).strip().lower()
    if pres_norm == "no":
        val = 0.0
    elif pres_norm == "yes":
        txt = "" if pd.isna(row.get("fireplace")) else str(row["fireplace"]).lower()

        # нормализуем словесные числа и извлекаем все числа
        s = txt
        for w, d in WORDS2NUM.items():
            s = re.sub(rf"\b{w}\b", d, s)
        nums = re.findall(r"\d+(?:\.\d+)?", s)

        if nums:
            # берём максимально упомянутое число (как наиболее вероятное количество)
            val = max(float(n) for n in nums)
        else:
            # без числа: пробуем по локациям
            locs = sum(bool(pat.search(txt)) for pat in LOC_PATTERNS.values())
            if locs >= 1:
                val = float(max(1, locs))  # минимум 1
            elif ONE_FP_HINT.search(txt):
                val = 1.0
            elif re.search(r"\bmultiple\s+fireplace(s)?\b", txt):
                val = 2.0
            else:
                val = 1.0  # дефолт для "есть камин", но деталей нет
    else:
        return np.nan

    return int(val)

In [1013]:
clean_fireplace["fireplace_count"] = clean_fireplace.apply(get_fireplace_count, axis=1)
clean_fireplace["fireplace_count"].value_counts()

fireplace_count
1.0       93166
2.0        2707
0.0        2555
3.0         722
4.0         233
5.0          75
6.0          41
7.0          20
8.0           5
9.0           3
10.0          3
11.0          2
1000.0        1
12.0          1
134.0         1
Name: count, dtype: int64

In [1014]:
clean_data['fireplace_count'] = clean_fireplace['fireplace_count']
clean_data['fireplace_location'] = clean_fireplace['fireplace_location']
clean_data['fireplace_type'] = clean_fireplace['fireplace_type']
clean_data['fireplace_present'] = clean_fireplace['fireplace_present']

#### 3.9 Очистим значения для колонки `stories`

In [1015]:
clean_stories = clean_data.copy()

In [1016]:
clean_stories['stories'].value_counts()

stories
1.0                     67324
2.0                     55214
1                       20746
2                       17087
3.0                     11257
                        ...  
4.0000                      1
1.2                         1
Bedroom - Split Plan        1
78                          1
65.0                        1
Name: count, Length: 343, dtype: int64

In [1017]:
def stories_to_number(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()

    # заменяем словесные целые на цифры (только целые!)
    tokens = s.split()
    for i,t in enumerate(tokens):
        if t in WORDS2NUM:
            tokens[i] = str(WORDS2NUM[t])
    s = " ".join(tokens)

    # извлекаем все числовые токены (3+ -> 3; '2 to 3' даст 2 и 3)
    nums = re.findall(r"\d+(?:\.\d+)?", s)
    if not nums:
        return np.nan

    # берём максимум (для '3+', '2-3', '2 to 3' и т.п.)
    val_num = max(float(n) for n in nums)
    return val_num

In [1018]:
clean_stories['stories_num'] = clean_stories['stories'].apply(stories_to_number)
clean_stories.loc[
    clean_stories['stories_num'] % 0.5 != 0,
    'stories_num'
] = np.nan
clean_stories['stories_num'].value_counts()

stories_num
1.0       96729
2.0       80034
3.0       17452
0.0       11508
9.0        3381
          ...  
82.0          1
1120.0        1
1002.0        1
96.0          1
65.0          1
Name: count, Length: 80, dtype: int64

In [1019]:
clean_data['stories_num'] = clean_stories['stories_num']

#### 3.10 Очистим значения для колонки `homeFacts`

In [1020]:
clean_home_facts = clean_data.copy()

In [1021]:
def extract_homefacts(row):
    """Достаёт значения по ключам factLabel"""
    result = {
        'Cooling': np.nan,
        'Heating': np.nan,
        'Parking': np.nan,
        'Price/sqft': np.nan,
        'Remodeled year': np.nan,
        'Year built': np.nan,
        'lotsize': np.nan
    }
    if row:
        data = ast.literal_eval(row)
        facts = data.get('atAGlanceFacts', [])
        for fact in facts:
            label = fact.get('factLabel')
            value = fact.get('factValue', np.nan) or np.nan
            if label:
                result[label] = value
    return pd.Series(result)

In [1022]:
facts_df = clean_home_facts['homeFacts'].apply(extract_homefacts)

In [1023]:
facts_df.head(10)

,Cooling,Heating,Parking,Price/sqft,Remodeled year,Year built,lotsize
0,NaN,"Central A/C, Heat Pump",NaN,$144,NaN,2019,NaN
1,NaN,NaN,NaN,$159/sqft,NaN,2019,5828 sqft
2,Central,Forced Air,Attached Garage,$965/sqft,1967,1961,"8,626 sqft"
3,Central,Forced Air,Detached Garage,$371/sqft,2006,2006,"8,220 sqft"
4,NaN,NaN,NaN,NaN,NaN,NaN,"10,019 sqft"
5,Central,Forced Air,NaN,$233/sqft,NaN,1920,680 sqft
6,Central Air,"Electric, Heat Pump",NaN,$120 / Sq. Ft.,2006,2006,"4,996 Sq. Ft."
7,NaN,NaN,NaN,$57 / Sq. Ft.,NaN,1976,"8,750 Sq. Ft."
8,Central,Forced Air,NaN,$68,NaN,1970,124582
9,Central,Gas,Attached Garage,$162,NaN,2019,"2,056 sqft"


In [1024]:
clean_data = pd.concat([clean_data, facts_df], axis=1)

#### 3.11 Очистим колонку Cooling

In [1025]:
# Analyze cooling column values
print("Cooling column value counts:")
print(clean_data['Cooling'].value_counts(dropna=False))
print(f"\nTotal unique values: {clean_data['Cooling'].nunique()}")
print(f"Missing values: {clean_data['Cooling'].isna().sum()} ({clean_data['Cooling'].isna().sum()/len(clean_data)*100:.1f}%)")

Cooling column value counts:
Cooling
Central                                                                158155
NaN                                                                    119877
Central Air                                                             12475
No Data                                                                 10616
Has Cooling                                                              9718
                                                                        ...  
Central Gas, Propane, Zoned                                                 1
Other (See Remarks), Panel/Floor/Wall, Window Unit                          1
Multi Units, Zoned Cooling                                                  1
Central Air, g-Energy Star HVAC, Gas Hot Air/Furnace, Multizone A/C         1
Central A/C (Gas), Central Heat (Gas), Heat Pump                            1
Name: count, Length: 1393, dtype: int64

Total unique values: 1392
Missing values: 119877 (32.7%)


In [1026]:
# Improved cooling column cleaning
def clean_cooling_type(val):
    if pd.isna(val) or val == '':
        return np.nan
    
    val_lower = str(val).lower().strip()
    
    # Central air conditioning (most common)
    if any(keyword in val_lower for keyword in ['central air', 'central a/c', 'central']):
        return 'central_air'
    
    # Window/wall units
    if any(keyword in val_lower for keyword in ['window', 'wall unit', 'room air']):
        return 'window_wall'
    
    # Heat pump (can be cooling too)
    if 'heat pump' in val_lower:
        return 'heat_pump'
    
    # Electric cooling
    if any(keyword in val_lower for keyword in ['electric', 'electrical']):
        return 'electric'
    
    # Gas cooling (less common but exists)
    if any(keyword in val_lower for keyword in ['gas', 'natural gas']):
        return 'gas'
    
    # Evaporative/swamp cooler
    if any(keyword in val_lower for keyword in ['evap', 'swamp', 'cooler']):
        return 'evaporative'
    
    # Ductless/mini-split
    if any(keyword in val_lower for keyword in ['ductless', 'mini split', 'split system']):
        return 'ductless'
    
    # Geothermal
    if any(keyword in val_lower for keyword in ['geothermal', 'ground source']):
        return 'geothermal'
    
    # No cooling/none
    if any(keyword in val_lower for keyword in ['none', 'no cooling', 'not applicable', 'n/a']):
        return 'none'
    
    # Other/unknown
    return 'other'

# Apply cleaning functions
clean_data['cooling_type'] = clean_data['Cooling'].apply(clean_cooling_type)

print("Cleaned cooling types:")
print(clean_data['cooling_type'].value_counts())

Cleaned cooling types:
cooling_type
central_air    199555
other           33972
none             7418
electric         1516
evaporative      1182
heat_pump        1178
gas               989
window_wall       968
ductless          165
geothermal         51
Name: count, dtype: int64


In [1027]:
def extract_cooling_energy_source(val):
    """Extract the energy source for cooling systems"""
    if pd.isna(val) or val == '':
        return np.nan

    val_lower = str(val).lower().strip()

    # Electric
    if any(keyword in val_lower for keyword in ['electric', 'electrical']):
        return 'electric'

    # Gas
    if any(keyword in val_lower for keyword in ['gas', 'natural gas', 'propane']):
        return 'gas'

    # Heat pump (typically electric but different mechanism)
    if 'heat pump' in val_lower:
        return 'heat_pump'

    # Geothermal
    if any(keyword in val_lower for keyword in ['geothermal', 'ground source']):
        return 'geothermal'

    return 'unknown'

clean_data['cooling_energy_source'] = clean_data['Cooling'].apply(extract_cooling_energy_source)
print("\nCooling energy sources:")
print(clean_data['cooling_energy_source'].value_counts())


Cooling energy sources:
cooling_energy_source
unknown       223031
electric       14164
gas             7917
heat_pump       1761
geothermal       121
Name: count, dtype: int64


#### 3.12 Очистим колонку Heating

In [1028]:
# Analyze heating column values
print("Heating column value counts:")
print(clean_data['Heating'].value_counts(dropna=False))
print(f"\nTotal unique values: {clean_data['Heating'].nunique()}")
print(f"Missing values: {clean_data['Heating'].isna().sum()} ({clean_data['Heating'].isna().sum()/len(clean_data)*100:.1f}%)")

Heating column value counts:
Heating
NaN                                                             105661
Forced Air                                                       81181
Forced air                                                       51462
Other                                                            29563
Electric                                                         10028
                                                                 ...  
3rd Floor bonus room-gas                                             1
Steam, Baseboard - Electric, Radiator                                1
On demand gas                                                        1
Attic Fan(s), Central Air, Central Heat, Fireplace(s)                1
Baseboard, Hot Water, Programmable Thermostat, Radiant Floor         1
Name: count, Length: 1839, dtype: int64

Total unique values: 1838
Missing values: 105661 (28.8%)


In [1029]:
# Clean heating column
def clean_heating_type(val):
    if pd.isna(val) or val == '':
        return np.nan

    val_lower = str(val).lower().strip()

    # Forced air (most common)
    if any(keyword in val_lower for keyword in ['forced air', 'forced-air']):
        return 'forced_air'

    # Central heating
    if any(keyword in val_lower for keyword in ['central', 'central heat']):
        return 'central'

    # Heat pump
    if 'heat pump' in val_lower:
        return 'heat_pump'

    # Radiant heating
    if any(keyword in val_lower for keyword in ['radiant', 'floor heat', 'in-floor']):
        return 'radiant'

    # Baseboard heating
    if any(keyword in val_lower for keyword in ['baseboard', 'base board']):
        return 'baseboard'

    # Boiler/hydronic
    if any(keyword in val_lower for keyword in ['boiler', 'hydronic', 'hot water']):
        return 'boiler'

    # Wall heaters
    if any(keyword in val_lower for keyword in ['wall heat', 'wall unit']):
        return 'wall_unit'

    # Space heaters
    if any(keyword in val_lower for keyword in ['space heat', 'room heat']):
        return 'space_heater'

    # Geothermal
    if any(keyword in val_lower for keyword in ['geothermal', 'ground source']):
        return 'geothermal'

    # No heating
    if any(keyword in val_lower for keyword in ['none', 'no heat', 'not applicable', 'n/a']):
        return 'none'

    # Other/unknown
    return 'other'

# Apply cleaning functions
clean_data['heating_type'] = clean_data['Heating'].apply(clean_heating_type)

print("Cleaned heating types:")
print(clean_data['heating_type'].value_counts())

Cleaned heating types:
heating_type
forced_air      138634
other            74056
central          32022
heat_pump        10007
baseboard         4149
radiant           1684
boiler             298
wall_unit          218
none                86
space_heater        31
geothermal          25
Name: count, dtype: int64


In [1030]:
def extract_heating_energy_source(val):
    """Extract the energy source for heating systems"""
    if pd.isna(val) or val == '':
        return np.nan

    val_lower = str(val).lower().strip()

    # Gas (natural gas, propane)
    if any(keyword in val_lower for keyword in ['gas', 'natural gas', 'propane', 'lng']):
        return 'gas'

    # Electric
    if any(keyword in val_lower for keyword in ['electric', 'electrical']):
        return 'electric'

    # Oil
    if any(keyword in val_lower for keyword in ['oil', 'fuel oil', 'heating oil']):
        return 'oil'

    # Heat pump (typically electric but different mechanism)
    if 'heat pump' in val_lower:
        return 'heat_pump'

    # Wood/biomass
    if any(keyword in val_lower for keyword in ['wood', 'pellet', 'biomass']):
        return 'wood'

    # Coal
    if 'coal' in val_lower:
        return 'coal'

    # Geothermal
    if any(keyword in val_lower for keyword in ['geothermal', 'ground source']):
        return 'geothermal'

    # Solar
    if 'solar' in val_lower:
        return 'solar'

    return 'unknown'

clean_data['heating_energy_source'] = clean_data['Heating'].apply(extract_heating_energy_source)
print("\nHeating energy sources:")
print(clean_data['heating_energy_source'].value_counts())


Heating energy sources:
heating_energy_source
unknown       204943
electric       26203
gas            17769
heat_pump      11833
oil              297
wood             106
geothermal        30
solar             29
Name: count, dtype: int64


#### 3.13 Очистим колонку Parking

In [1031]:
# Analyze parking column values
print("Parking column value counts:")
print(clean_data['Parking'].value_counts(dropna=False))
print(f"\nTotal unique values: {clean_data['Parking'].nunique()}")
print(
    f"Missing values: {clean_data['Parking'].isna().sum()} ({clean_data['Parking'].isna().sum() / len(clean_data) * 100:.1f}%)")

Parking column value counts:
Parking
NaN                                                                              169457
Attached Garage                                                                   70549
2 spaces                                                                          28045
1 space                                                                           14238
No Data                                                                           13331
                                                                                  ...  
Carport, On Street, Detached Garage, Off Street                                       1
Carport - 1, Covered Parking, Off Street Parking, Guest Parking, Parking Area         1
Assigned Parking Space - 1, Parking Garage, Parking Space - 2                         1
On-site - Rent, Parking Fee                                                           1
Paved Driveway, Off Street, Detached Garage                                        

In [1032]:
# Define parking type cleaning function
def clean_parking_type(val):
    if pd.isna(val) or val == '' or str(val).strip() == '':
        return np.nan
    
    val_lower = str(val).lower().strip()
    
    # Attached garage (most desirable)
    if any(keyword in val_lower for keyword in ['attached garage', 'attached']):
        return 'attached_garage'
    
    # Detached garage
    if any(keyword in val_lower for keyword in ['detached garage', 'detached']):
        return 'detached_garage'
    
    # General garage (when type not specified)
    if 'garage' in val_lower and 'attached' not in val_lower and 'detached' not in val_lower:
        return 'garage'
    
    # Carport
    if any(keyword in val_lower for keyword in ['carport', 'car port']):
        return 'carport'
    
    # Driveway only
    if any(keyword in val_lower for keyword in ['driveway', 'drive']):
        return 'driveway'
    
    # Street parking
    if any(keyword in val_lower for keyword in ['street', 'on street', 'street parking']):
        return 'street'
    
    # Covered parking
    if any(keyword in val_lower for keyword in ['covered', 'covered parking']):
        return 'covered'
    
    # Uncovered parking
    if any(keyword in val_lower for keyword in ['uncovered', 'open']):
        return 'uncovered'
    
    # RV parking
    if any(keyword in val_lower for keyword in ['rv', 'recreational vehicle']):
        return 'rv_parking'
    
    # No parking
    if any(keyword in val_lower for keyword in ['none', 'no parking', 'not applicable', 'n/a']):
        return 'none'
    
    # Other/unknown
    return 'other'

In [1033]:
# Define parking spaces extraction function
def extract_parking_spaces(val):
    """Extract the number of parking spaces"""
    if pd.isna(val) or val == '' or str(val).strip() == '':
        return 0
    
    val_str = str(val).lower().strip()
    
    # Extract numbers from the text
    numbers = re.findall(r'\b(\d+)\b', val_str)
    
    if numbers:
        # Take the first number found (usually the number of spaces)
        return int(numbers[0])
    
    # If no number found, try to infer from keywords
    if any(keyword in val_str for keyword in ['single', 'one', '1']):
        return 1
    elif any(keyword in val_str for keyword in ['double', 'two', '2']):
        return 2
    elif any(keyword in val_str for keyword in ['triple', 'three', '3']):
        return 3
    elif 'garage' in val_str or 'carport' in val_str:
        return 1  # Default assumption for garage/carport
    
    return 0  # Default for unknown

In [1034]:
# Define garage detection function
def has_garage(val):
    """Check if property has any type of garage"""
    if pd.isna(val) or val == '' or str(val).strip() == '':
        return False
    
    val_lower = str(val).lower().strip()
    return 'garage' in val_lower

In [1035]:
# Apply parking cleaning functions
clean_data['parking_type'] = clean_data['Parking'].apply(clean_parking_type)
clean_data['parking_spaces'] = clean_data['Parking'].apply(extract_parking_spaces)
clean_data['has_garage'] = clean_data['Parking'].apply(has_garage)

In [1036]:
# Display parking cleaning results
print("Cleaned parking types:")
print(clean_data['parking_type'].value_counts())
print("\nParking spaces distribution:")
print(clean_data['parking_spaces'].value_counts())
print(f"\nProperties with garage: {clean_data['has_garage'].sum()} ({clean_data['has_garage'].mean()*100:.1f}%)")

Cleaned parking types:
parking_type
attached_garage    82013
other              76282
detached_garage    14834
carport             8452
street              8332
garage              2957
none                2370
driveway            1372
uncovered            441
rv_parking           244
covered              117
Name: count, dtype: int64

Parking spaces distribution:
parking_spaces
0       198299
1       123581
2        32863
3         5473
4         3601
         ...  
93           1
51           1
2020         1
73           1
118          1
Name: count, Length: 93, dtype: int64

Properties with garage: 97881 (26.7%)


#### 3.14 Очистим колонки Year built и Remodeled year

In [1037]:
# Analyze Year built and Remodeled year columns
print("Year built column analysis:")
print(f"Unique values: {clean_data['Year built'].nunique()}")
print(
    f"Missing values: {clean_data['Year built'].isna().sum()} ({clean_data['Year built'].isna().sum() / len(clean_data) * 100:.1f}%)")
print("Sample values:")
print(clean_data['Year built'].value_counts().head(10))

print("\n" + "=" * 50 + "\n")

print("Remodeled year column analysis:")
print(f"Unique values: {clean_data['Remodeled year'].nunique()}")
print(
    f"Missing values: {clean_data['Remodeled year'].isna().sum()} ({clean_data['Remodeled year'].isna().sum() / len(clean_data) * 100:.1f}%)")
print("Sample values:")
print(clean_data['Remodeled year'].value_counts().head(10))

Year built column analysis:
Unique values: 227
Missing values: 61741 (16.8%)
Sample values:
Year built
2019    31052
2006     7789
2005     7276
2007     6989
2018     6704
2004     5350
2017     5102
2016     4991
2008     4881
1950     4441
Name: count, dtype: int64


Remodeled year column analysis:
Unique values: 153
Missing values: 216135 (58.9%)
Sample values:
Remodeled year
2006    5530
2005    4808
2007    4379
2008    3789
2004    3359
1980    3325
1970    3144
2000    3060
2003    2822
1985    2811
Name: count, dtype: int64


In [1038]:
# Clean year column
def clean_year(val):
    if pd.isna(val) or val == '' or str(val).strip() == '':
        return np.nan

    val_str = str(val).strip()

    # Extract 4-digit year
    year_match = re.search(r'\b(19\d{2}|20\d{2})\b', val_str)

    if year_match:
        year = int(year_match.group(1))
        return year

    # Try to extract any 4-digit number
    numbers = re.findall(r'\b\d{4}\b', val_str)
    if numbers:
        year = int(numbers[0])
        return year

    return np.nan

In [1039]:
# Apply year cleaning functions
clean_data['year_built'] = clean_data['Year built'].apply(clean_year)
clean_data['remodeled_year'] = clean_data['Remodeled year'].apply(clean_year)

In [1040]:
# Display year cleaning results
print("Year built cleaning results:")
print(f"Original missing: {clean_data['Year built'].isna().sum()}")
print(f"Cleaned missing: {clean_data['year_built'].isna().sum()}")
print(f"Year range: {clean_data['year_built'].min():.0f} - {clean_data['year_built'].max():.0f}")
print("Year built distribution:")
print(clean_data['year_built'].describe())

print("\n" + "=" * 50 + "\n")

print("Remodeled year cleaning results:")
print(f"Original missing: {clean_data['Remodeled year'].isna().sum()}")
print(f"Cleaned missing: {clean_data['remodeled_year'].isna().sum()}")

if clean_data['remodeled_year'].notna().sum() > 0:
    print(f"Remodel year range: {clean_data['remodeled_year'].min():.0f} - {clean_data['remodeled_year'].max():.0f}")
    print("Remodeled year distribution:")
    print(clean_data['remodeled_year'].describe())

Year built cleaning results:
Original missing: 61741
Cleaned missing: 62967
Year range: 1019 - 2025
Year built distribution:
count    303904.000000
mean       1979.229151
std          33.743809
min        1019.000000
25%        1956.000000
50%        1985.000000
75%        2007.000000
max        2025.000000
Name: year_built, dtype: float64


Remodeled year cleaning results:
Original missing: 216135
Cleaned missing: 216405
Remodel year range: 1111 - 2021
Remodeled year distribution:
count    150466.000000
mean       1982.710891
std          25.037841
min        1111.000000
25%        1968.000000
50%        1986.000000
75%        2004.000000
max        2021.000000
Name: remodeled_year, dtype: float64


#### 3.15 Очистим колонку lotsize

In [1041]:
# Analyze lotsize column values
print("Lotsize column value counts:")
print(clean_data['lotsize'].value_counts(dropna=False).head(20))
print(f"\nTotal unique values: {clean_data['lotsize'].nunique()}")
print(
    f"Missing values: {clean_data['lotsize'].isna().sum()} ({clean_data['lotsize'].isna().sum() / len(clean_data) * 100:.1f}%)")

Lotsize column value counts:
lotsize
NaN            59860
—              25171
No Data         5329
-- sqft lot     3819
0.26 acres      2503
0.25 acres      2335
0.28 acres      2093
0.27 acres      1935
0.29 acres      1914
0.34 acres      1602
6,098 sqft      1523
0.3 acres       1436
7,405 sqft      1357
0.31 acres      1335
6,534 sqft      1288
0.32 acres      1272
4,356 sqft      1269
10,000 sqft     1256
5,227 sqft      1173
5,000 sqft      1115
Name: count, dtype: int64

Total unique values: 35812
Missing values: 59860 (16.3%)


In [1042]:
# Clean lotsize column function
def clean_lotsize(val):
    if pd.isna(val) or val == '' or str(val).strip() == '':
        return np.nan

    val_str = str(val).lower().strip()

    # Remove common text and clean up
    val_str = val_str.replace(',', '').replace('lot', '').replace('size', '').strip()

    # Handle acres - convert to square feet (1 acre = 43,560 sqft)
    if 'acre' in val_str:
        # Extract number before 'acre'
        acre_match = re.search(r'([\d,.]+)\s*acre', val_str)
        if acre_match:
            acres = float(acre_match.group(1).replace(',', ''))
            return int(acres * 43560)  # Convert to square feet

    # Handle square feet
    if 'sqft' in val_str or 'sq ft' in val_str or 'sq. ft' in val_str:
        # Extract number before sqft
        sqft_match = re.search(r'([\d,]+)\s*(?:sqft|sq\s*ft|sq\.\s*ft)', val_str)
        if sqft_match:
            return int(sqft_match.group(1).replace(',', ''))

    # Try to extract any number (assume it's square feet)
    numbers = re.findall(r'\b([\d,]+)\b', val_str)
    if numbers:
        try:
            size = int(numbers[0].replace(',', ''))
            return size
        except ValueError:
            pass

    return np.nan

In [1043]:
# Apply lotsize cleaning function
clean_data['lotsize_sqft'] = clean_data['lotsize'].apply(clean_lotsize)

In [1044]:
# Display lotsize cleaning results
print("Lotsize cleaning results:")
print(f"Original missing: {clean_data['lotsize'].isna().sum()}")
print(f"Cleaned missing: {clean_data['lotsize_sqft'].isna().sum()}")

if clean_data['lotsize_sqft'].notna().sum() > 0:
    print(f"Lot size range: {clean_data['lotsize_sqft'].min():.0f} - {clean_data['lotsize_sqft'].max():.0f} sqft")
    print("Lot size distribution:")
    print(clean_data['lotsize_sqft'].describe())

Lotsize cleaning results:
Original missing: 59860
Cleaned missing: 94179
Lot size range: 0 - 2147483647 sqft
Lot size distribution:
count    2.726920e+05
mean     1.307091e+05
std      9.203064e+06
min      0.000000e+00
25%      5.227000e+03
50%      8.276000e+03
75%      1.481000e+04
max      2.147484e+09
Name: lotsize_sqft, dtype: float64


#### 3.16 Очистим целевую переменную target (цена)

In [1045]:
# Analyze target column (price)
print("Target column analysis:")
print(f"Total records: {len(clean_data)}")
print(f"Missing values: {clean_data['target'].isna().sum()}")
print(f"Data type: {clean_data['target'].dtype}")
print("\nSample values:")
print(clean_data['target'].value_counts().head(10))
print("\nUnique price formats (first 20):")
unique_prices = clean_data['target'].unique()[:20]
for price in unique_prices:
    if pd.notna(price):
        print(f"'{price}'")

Target column analysis:
Total records: 366871
Missing values: 2174
Data type: object

Sample values:
target
$225,000    1413
$275,000    1311
$250,000    1271
$350,000    1261
$299,900    1232
$399,000    1208
$325,000    1206
$249,900    1188
$299,000    1150
$199,900    1141
Name: count, dtype: int64

Unique price formats (first 20):
'$418,000'
'$310,000'
'$2,895,000'
'$2,395,000'
'$5,000'
'$209,000'
'181,500'
'68,000'
'$244,900'
'$311,995'
'$669,000'
'260,000'
'$525,000'
'$499,900'
'$168,800'
'1,650,000'
'335,000'
'2,650,000'
'$365,000'
'$626,000'


In [1046]:
# Clean target price column function
def clean_target_price(val):
    if pd.isna(val) or val == '' or str(val).strip() == '':
        return np.nan

    val_str = str(val).strip()

    # Remove dollar sign and common formatting
    val_str = val_str.replace('$', '').replace(',', '').strip()

    # Handle special cases
    if val_str.lower() in ['', 'nan', 'none', 'n/a', 'not available']:
        return np.nan

    # Extract numbers (handle cases like "418000", "418,000", etc.)
    numbers = re.findall(r'\d+', val_str)

    if numbers:
        # Join all numbers (in case price was split by commas)
        price_str = ''.join(numbers)
        price = int(price_str)
        return price

    return np.nan

In [1047]:
# Apply price cleaning function
clean_data['target'] = clean_data['target'].apply(clean_target_price)

#### 3.17 Очистим колонку zipcode

In [1048]:
# Analyze zipcode column
print("Zipcode column analysis:")
print(f"Data type: {clean_data['zipcode'].dtype}")
print(f"Missing values: {clean_data['zipcode'].isna().sum()}")
print(f"Unique values: {clean_data['zipcode'].nunique()}")

print("\nSample zipcode values:")
print(clean_data['zipcode'].value_counts().head(10))

print("\nZipcode length analysis:")
zipcode_lengths = clean_data['zipcode'].astype(str).str.len().value_counts().sort_index()
print(zipcode_lengths)

Zipcode column analysis:
Data type: object
Missing values: 0
Unique values: 4530

Sample zipcode values:
zipcode
32137    1985
33131    1550
78245    1389
34747    1377
33132    1324
33137    1305
78253    1282
78254    1238
33130    1167
34759    1143
Name: count, dtype: int64

Zipcode length analysis:
zipcode
1          3
2          2
4       1843
5     364780
6          3
8          1
9          1
10       238
Name: count, dtype: int64


In [1049]:
# Clean zipcode function - keep as string for ML modeling
def clean_zipcode(val):
    if pd.isna(val) or val == '' or str(val).strip() == '':
        return np.nan

    val_str = str(val).strip()

    # Handle ZIP+4 format (e.g., "27603-5569" -> "27603")
    if '-' in val_str:
        # Take only the first 5 digits (main ZIP code)
        val_str = val_str.split('-')[0]

    # Remove any non-numeric characters except leading zeros
    cleaned = re.sub(r'[^0-9]', '', val_str)

    if not cleaned:
        return np.nan

    # Convert to integer first for validation
    try:
        zipcode_int = int(cleaned)

        # Validate US zipcode range (general validation)
        # US zipcodes are typically 5 digits: 00501 to 99950
        if 501 <= zipcode_int <= 99999:
            # Return as 5-digit string with leading zeros
            return f"{zipcode_int:05d}"
        else:
            # Handle edge cases like leading zeros that got dropped
            if len(cleaned) == 4 and zipcode_int >= 501:
                # Return as 4-digit string (valid short zipcodes)
                return f"{zipcode_int:04d}"
            elif len(cleaned) == 3 and zipcode_int >= 51:
                # Return as 3-digit string (rare but possible)
                return f"{zipcode_int:03d}"
    except ValueError:
        pass

    return np.nan

In [1050]:
# Apply zipcode cleaning function - returns string format
clean_data['zipcode'] = clean_data['zipcode'].apply(clean_zipcode)
clean_data = clean_data.dropna(subset=['zipcode'])

#### 3.18 Валидация и очистка колонки state

In [1051]:
# Define valid US states and territories
valid_us_states = {
    # 50 States
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
    'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
    'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
    'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
    'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY',
    # Federal District
    'DC',
    # Territories
    'AS', 'GU', 'MP', 'PR', 'VI'
}

In [1052]:
clean_data['state'] = clean_data['state'].str.upper()
current_states = set(clean_data['state'].dropna().unique())
invalid_states_found = current_states - set(valid_us_states)
print(invalid_states_found)

{'BA', 'OT'}


In [1053]:
clean_data.loc[clean_data['state'].isin(invalid_states_found), 'state'] = 'unknown'

#### 3.19 Удалим ненужные колонки

In [1054]:
# Transform Price/sqft column to numeric
print("Transforming Price/sqft column to numeric...")
print("=" * 50)

# Check current Price/sqft column
print("Current Price/sqft analysis:")
print(f"Data type: {clean_data['Price/sqft'].dtype}")
print(f"Missing values: {clean_data['Price/sqft'].isna().sum()}")
print(f"Total records: {len(clean_data)}")

Transforming Price/sqft column to numeric...
Current Price/sqft analysis:
Data type: object
Missing values: 62672
Total records: 366862


In [1055]:
def clean_price_per_sqft(val):
    """Convert Price/sqft to numeric value"""
    if pd.isna(val) or val == '' or str(val).strip() == '':
        return np.nan

    val_str = str(val).strip()

    # Remove common text and symbols
    val_str = val_str.replace('$', '').replace('/sqft', '').replace('/sq.ft', '')
    val_str = val_str.replace('/Sq. Ft.', '').replace('/ Sq. Ft.', '').replace(',', '')
    val_str = val_str.strip()

    # Extract numeric value
    numbers = re.findall(r'\d+(?:\.\d+)?', val_str)

    if numbers:
        # Take the first number found
        price_per_sqft = float(numbers[0])
        return price_per_sqft

    return np.nan

In [1056]:
# Apply transformation
clean_data['price_per_sqft'] = clean_data['Price/sqft'].apply(clean_price_per_sqft)

In [1057]:
# Check new Price/sqft column
print("New Price/sqft analysis:")
print(f"Data type: {clean_data['price_per_sqft'].dtype}")
print(f"Missing values: {clean_data['price_per_sqft'].isna().sum()}")
print(f"Total records: {len(clean_data)}")

New Price/sqft analysis:
Data type: float64
Missing values: 64689
Total records: 366862


#### 3.20 Удалим ненужные колонки

In [1058]:
clean_data = clean_data.drop(columns=['baths', 'homeFacts', 'fireplace', 'sqft', 'beds', 'stories', 'mls-id', 'MlsId', 'Cooling', 'Heating', 'Parking', 'Year built', 'Remodeled year', 'lotsize', 'city', 'Price/sqft'])

In [1059]:
clean_data.to_csv('cleaned_housing_data.csv', index=False)